# 描述性统计

## 导入库

In [12]:
import os
import dotenv
import re
import warnings
import polars as pl
import plotly  
import plotly.express as px
from plotly.subplots import make_subplots
import statsmodels.api as sm
dotenv.load_dotenv()

True

## 超参数

In [13]:
# 基本配置
BASELINE_TASK_ID_PREFIX = 'mech1'  # 基线任务id前缀
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{BASELINE_TASK_ID_PREFIX}' # 保存基本路
SAVE = True # 是否保存数据

# 数据库
CONNECTION_URL = os.getenv("POSTGRES_URL")
ENGINE = "connectorx"



## 读取数据  
1.FM回归的控制变量  
2.MA数据

In [14]:
fm_reg = pl.read_database_uri(
    """SELECT * 
    FROM statics.fm_reg_controls""",
    uri = CONNECTION_URL,
    engine = ENGINE
)


In [15]:
fm_describe = fm_reg.select(pl.all().exclude(['stkcd','accper'])).describe()

In [16]:
ma_describe = pl.read_parquet(f"{SAVE_BASE_DIR}/MA因子_copy.parquet").select(pl.col(['MA', 'return'])).describe()

In [17]:
ma_describe

statistic,MA,return
str,f64,f64
"""count""",388093.0,388093.0
"""null_count""",0.0,0.0
"""mean""",0.457123,0.005242
"""std""",0.215294,0.144232
"""min""",-0.5,-0.836998
"""25%""",0.325037,-0.0738
"""50%""",0.47728,-0.005133
"""75%""",0.605667,0.0714
"""max""",1.3971,4.150268


In [18]:
desc =ma_describe.join(fm_describe, on = 'statistic', how = 'left')

In [19]:
# 先 unpivot：统计量保留为 index，各变量列压成 variable + value
long = desc.unpivot(
    index="statistic",
    on=[c for c in desc.columns if c != "statistic"],  # 或直接写 ["MA", "return", "betavals", ...]
    variable_name="variable",
    value_name="value"
)

# 再 pivot：行=变量，列=统计量
result = long.pivot(
    values="value",
    index="variable",
    on = "statistic"
)

result

variable,count,null_count,mean,std,min,25%,50%,75%,max
str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""MA""",388093.0,0.0,0.457123,0.215294,-0.5,0.325037,0.47728,0.605667,1.3971
"""return""",388093.0,0.0,0.005242,0.144232,-0.836998,-0.0738,-0.005133,0.0714,4.150268
"""betavals""",998924.0,1076.0,1.11061,5.73821,-2376.66667,0.72842,1.10061,1.4775,2540.83333
"""bm_ratio""",325753.0,674247.0,0.650698,0.264081,-0.221668,0.464737,0.65581,0.83641,43.587682
"""gross_margin""",318602.0,681398.0,0.265466,3.826124,-2137.8044,0.1505,0.2446,0.3714,2.1605
"""investment_ratio""",402584.0,597416.0,0.942977,0.103233,-0.310563,0.93515,0.989542,0.999999,1.23341
"""ln_market_value""",999501.0,499.0,15.06076,1.320145,9.939355,14.187714,14.984791,15.818205,21.747961


In [20]:
result.write_parquet(f"/home/frank/files/programs/GraduationThesis/empirical/描述性统计.parquet")

## 附录-因子信息

In [21]:
factors = pl.read_database_uri(
    """SELECT * 
    FROM factors_data.factor_metadata""",
    uri = CONNECTION_URL,
    engine = ENGINE
)


In [22]:
cates = factors['category'].unique()